# 05 · Milestone 2 model integration

**IE 7615 · From Words to Worlds**

This notebook runs Stable Diffusion 1.5 on five COCO captions. Each caption goes through two paths: the pipeline encodes the text internally, and we explicitly encode the same text with the pipeline's CLIP components. We compare the conditioning tensors and denoised latents, then save five baseline images and five companion images to Google Drive.

Open this notebook in Google Colab and select **Runtime → Change runtime type → GPU**. Run sections 01–08 in order and authorize the Drive mount when prompted. The experiment needs internet access for package and checkpoint downloads. The repository's caption manifests are sufficient; the COCO images are only needed for later work with real-image references.

## 01 · Set up the Colab GPU

Keep the CUDA build of PyTorch and its companion packages from the Colab runtime. This cell installs the four inference-library versions used by Milestone 1 and constrains pip to the runtime's existing PyTorch package versions. A small FP16 CUDA operation checks the GPU before any model download. If this cell requests a session restart, restart the session and run from section 01 again.

In [ ]:
import importlib.metadata as metadata
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from packaging.version import Version

if sys.version_info < (3, 11):
    raise RuntimeError("Select a Colab runtime with Python 3.11 or newer.")
if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError("Select a GPU under Runtime → Change runtime type and rerun this cell.")
if Version(torch.__version__).release < (2, 6):
    raise RuntimeError("Select a Colab GPU runtime with PyTorch 2.6 or newer.")

device = torch.device("cuda", torch.cuda.current_device())
dtype = torch.float16
try:
    probe = torch.ones((8, 8), device=device, dtype=dtype)
    if (probe @ probe)[0, 0].item() != 8.0:
        raise RuntimeError("The FP16 CUDA probe returned an unexpected result.")
    del probe
except RuntimeError as error:
    raise RuntimeError("Choose an NVIDIA GPU runtime with working FP16 support.") from error

inference_pins = {
    "diffusers": "0.42.1",
    "transformers": "5.22.1",
    "accelerate": "1.15.0",
    "safetensors": "0.8.0",
}
runtime_packages = {
    dist.metadata["Name"].lower(): dist.version
    for dist in metadata.distributions()
    if dist.metadata["Name"]
}
torch_packages = {
    name: runtime_packages[name]
    for name in ("torch", "torchvision", "torchaudio")
    if name in runtime_packages
}
if any(runtime_packages.get(name) != version for name, version in inference_pins.items()):
    was_imported = any(
        name in sys.modules
        for name in (*inference_pins, "huggingface_hub", "tokenizers")
    )
    with TemporaryDirectory() as temp_dir:
        constraints = Path(temp_dir) / "colab-torch.txt"
        constraints.write_text(
            "".join(f"{name}=={version}\n" for name, version in torch_packages.items()),
            encoding="utf-8",
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--constraint", str(constraints)]
            + [f"{name}=={version}" for name, version in inference_pins.items()],
            check=True,
        )
    if was_imported:
        raise RuntimeError("Inference libraries were already imported. Restart the Colab session and run again.")

for name, version in {**torch_packages, **inference_pins}.items():
    if metadata.version(name) != version:
        raise RuntimeError(f"{name} version differs from the expected {version}.")

import numpy as np
from diffusers import StableDiffusionPipeline
from PIL import Image

gpu = torch.cuda.get_device_properties(device)
driver_info = None
if nvidia_smi := shutil.which("nvidia-smi"):
    driver_query = subprocess.run(
        [nvidia_smi, "--query-gpu=index,name,driver_version", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    driver_info = driver_query.stdout.strip() or None
runtime_info = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "device": str(device),
    "gpu": gpu.name,
    "compute_capability": [gpu.major, gpu.minor],
    "gpu_total_memory_bytes": gpu.total_memory,
    "nvidia_smi": driver_info,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "packages": {
        name: metadata.version(name)
        for name in (*torch_packages, *inference_pins, "huggingface-hub", "tokenizers", "numpy", "pillow")
    },
}
print(json.dumps(runtime_info, indent=2))

## 02 · Locate the GitHub checkout

If a checkout already exists, set `REPO_DIR` to its location or run from its root. In a fresh Colab session, this cell clones the Milestone 1 branch to fetch the source modules and manifests. Opening a notebook from GitHub in Colab still requires this checkout step. If the clone is unavailable, section 03 explains how to supply the manifest or use the included examples.

In [ ]:
REPO_URL = "https://github.com/connor-rivera/ie7615-generative-project.git"
REPO_REF = "milestone1-complete"
REPO_DIR = Path("/content/ie7615-generative-project")
PROMPTS_JSON_OVERRIDE = None  # For an uploaded manifest, use Path("/content/prompts.json").

checkout_candidates = (REPO_DIR, Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path.resolve() for path in checkout_candidates if (path / "src/prompts.py").is_file()),
    None,
)
if repo_root is None and not REPO_DIR.exists():
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        capture_output=True,
        text=True,
        check=False,
    )
    if clone.returncode == 0 and (REPO_DIR / "src/prompts.py").is_file():
        repo_root = REPO_DIR.resolve()

repo_commit = None
if repo_root is not None:
    sys.path.insert(0, str(repo_root))
    git_revision = subprocess.run(
        ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=False,
    )
    if git_revision.returncode == 0:
        repo_commit = git_revision.stdout.strip()
    print(f"Checkout {repo_root} · commit {repo_commit or 'unavailable'}")
else:
    print("Checkout unavailable. Set REPO_DIR to an accessible checkout or use section 03's fallback.")

## 03 · Select five COCO captions

We use the five smallest COCO image IDs in `manifests/prompts.json`, matching notebook 04. The manifest already contains one seeded caption per clean image. If the prompt manifest is missing but the clean dataset manifest is available, we rebuild the prompt map in memory with `src.prompts.select_prompts`.

If the checkout is unavailable, upload the repository's `manifests/prompts.json` through Colab's **Files → Upload**, set `PROMPTS_JSON_OVERRIDE = Path("/content/prompts.json")` in section 02, and rerun sections 02–03. With neither manifest, the notebook uses the same five captions copied from Milestone 1 and records that source in the run metadata.

In [ ]:
import hashlib

if repo_root is not None:
    from src.data import MANIFEST_PATH, load_manifest
    from src.generate_samples import (
        DEFAULT_GUIDANCE_SCALE,
        DEFAULT_SEED,
        DEFAULT_STEPS,
        IMAGE_SIZE,
        MODEL_ID,
        NEGATIVE_PROMPT,
    )
    from src.prompts import PROMPT_SEED, PROMPTS_PATH, load_prompts, select_prompts
else:
    # These are the Milestone 1 constants for a standalone Colab upload.
    MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    DEFAULT_SEED, DEFAULT_STEPS, DEFAULT_GUIDANCE_SCALE, IMAGE_SIZE = 7615, 30, 7.5, 512
    NEGATIVE_PROMPT = "blurry, distorted, low quality, duplicate, text, watermark"
    PROMPT_SEED = 7615
    PROMPTS_PATH = None

prompt_path = Path(PROMPTS_JSON_OVERRIDE) if PROMPTS_JSON_OVERRIDE is not None else PROMPTS_PATH
if PROMPTS_JSON_OVERRIDE is not None and not prompt_path.is_file():
    raise FileNotFoundError(f"Upload prompts.json and correct PROMPTS_JSON_OVERRIDE: {prompt_path}")

if prompt_path is not None and prompt_path.is_file():
    prompt_bytes = prompt_path.read_bytes()
    prompt_document = json.loads(prompt_bytes)
    prompt_map = (
        load_prompts(prompt_path) if repo_root is not None
        else {entry["image_id"]: entry["prompt"] for entry in prompt_document["prompts"]}
    )
    if len(prompt_map) != len(prompt_document["prompts"]) or len(prompt_map) != prompt_document["n_prompts"]:
        raise ValueError("The prompt manifest has duplicate IDs or an inconsistent prompt count.")
    prompt_source = {
        "kind": "prompt_manifest",
        "path": str(prompt_path),
        "sha256": hashlib.sha256(prompt_bytes).hexdigest(),
        "selection_seed": prompt_document["seed"],
    }
elif repo_root is not None and MANIFEST_PATH.is_file():
    prompt_map = select_prompts(load_manifest(), seed=PROMPT_SEED)
    prompt_source = {
        "kind": "rebuilt_from_clean_manifest",
        "path": str(MANIFEST_PATH),
        "sha256": hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
        "selection_seed": PROMPT_SEED,
    }
else:
    prompt_map = {
        139: "A room with chairs, a table, and a woman in it.",
        285: "A large bear that is sitting on grass.",
        632: "Bedroom scene with a bookcase, blue comforter and window.",
        724: "A stop sign is mounted upside-down on it's post.",
        776: "Three stuffed bears hugging and sitting on a blue pillow",
    }
    prompt_source = {"kind": "embedded_milestone1_examples", "selection_seed": PROMPT_SEED}

if len(prompt_map) < 5 or any(type(image_id) is not int for image_id in prompt_map):
    raise ValueError("Expected at least five captions keyed by integer COCO image IDs.")
samples = sorted(prompt_map.items())[:5]
if any(not isinstance(caption, str) or not caption.strip() for _, caption in samples):
    raise ValueError("Each of the five selected captions must contain text.")
prompt_source["available_prompts"] = len(prompt_map)
prompt_source["selection_rule"] = "first five after sorting by COCO image ID"
print(f"Caption source: {prompt_source['kind']}")
for index, (image_id, caption) in enumerate(samples):
    print(f"COCO {image_id:012d} · seed {DEFAULT_SEED + index} · {caption}")

## 04 · Mount Drive and set the inference configuration

Authorize Colab to mount your Google Drive. A run gets its own timestamped folder under `MyDrive/IE7615/milestone2/model_integration/`. The checkpoint revision comes from the snapshot cached during Milestone 1, and the image size, guidance, negative prompt, and requested step count come from its generator. We also record the actual denoising calls because a scheduler can use more timesteps than requested.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive")
if not drive_root.is_dir():
    raise RuntimeError("Google Drive is unavailable. Authorize the mount and rerun this cell.")
output_root = drive_root / "IE7615" / "milestone2" / "model_integration"

MODEL_REVISION = "451f4fe16113bff5a5d2269ed5ad43b0592e9a14"
inference_settings = {
    "height": IMAGE_SIZE,
    "width": IMAGE_SIZE,
    "num_inference_steps": DEFAULT_STEPS,
    "guidance_scale": DEFAULT_GUIDANCE_SCALE,
    "num_images_per_prompt": 1,
    "clip_skip": None,
    "output_type": "pil",
}
embedding_tolerance = {"atol": 1e-3, "rtol": 1e-3}
latent_tolerance = {"atol": 2e-3, "rtol": 2e-3}
print(json.dumps(inference_settings, indent=2))

## 05 · Load Stable Diffusion 1.5 in FP16

The pipeline loads one checkpoint with its own CLIP tokenizer, CLIP text encoder, conditional UNet, VAE, and PNDM scheduler. CLIP produces one hidden-state vector per token. The UNet uses those vectors as cross-attention context during denoising, and the VAE decodes the denoised latent into a 512×512 image. The checkpoint's safety checker runs on both image paths.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=dtype,
    use_safetensors=True,
).to(device)
pipe.set_progress_bar_config(leave=False)
for module in (pipe.text_encoder, pipe.unet, pipe.vae):
    module.eval()
    parameter = next(module.parameters())
    if parameter.device != device or parameter.dtype != dtype:
        raise RuntimeError(f"{type(module).__name__} must be loaded in FP16 on {device}.")

components = {
    name: type(component).__name__ if component is not None else None
    for name, component in pipe.components.items()
}
scheduler_type = type(pipe.scheduler)
scheduler_config = dict(pipe.scheduler.config)
token_count = pipe.tokenizer.model_max_length
hidden_size = pipe.text_encoder.config.hidden_size
if (token_count, hidden_size, pipe.unet.config.cross_attention_dim) != (77, 768, 768):
    raise RuntimeError("The tokenizer, CLIP encoder, and UNet dimensions must match SD 1.5.")
if scheduler_type.__name__ != "PNDMScheduler":
    raise RuntimeError("The pinned SD 1.5 baseline expects its PNDM scheduler.")

latent_shape = (
    1,
    pipe.unet.config.in_channels,
    IMAGE_SIZE // pipe.vae_scale_factor,
    IMAGE_SIZE // pipe.vae_scale_factor,
)
attention_processors = sorted({type(processor).__name__ for processor in pipe.unet.attn_processors.values()})
print(f"Model {MODEL_ID} @ {MODEL_REVISION}")
print(json.dumps(components, indent=2))
print(f"UNet attention processors: {attention_processors}")
print(f"CLIP context: [batch, {token_count}, {hidden_size}] · initial latent shape: {latent_shape}")

## 06 · Encode text explicitly and compare with Diffusers

Pad and truncate to 77 tokens using the pipeline's tokenizer. Pass the attention mask when the CLIP config requests it, and pass `last_hidden_state` to diffusion. With classifier-free guidance (CFG) at 7.5, we encode both the caption and the shared Milestone 1 negative prompt.

Each comparison uses a one-caption batch, matching the generation calls. Compare the explicit and internal results from `pipe.encode_prompt` in FP32 for measurement. The tensors supplied to the UNet stay in FP16. We record exact equality, maximum and mean absolute error, and whether each tensor passes the stated tolerances.

In [ ]:
def tensor_agreement(actual, reference, tolerance):
    if actual.shape != reference.shape:
        raise ValueError(f"Tensor shape mismatch: {tuple(actual.shape)} vs {tuple(reference.shape)}")
    actual, reference = actual.float(), reference.float()
    if not (torch.isfinite(actual).all().item() and torch.isfinite(reference).all().item()):
        return {"finite": False, "within_tolerance": False, "exact": False,
                "max_abs_error": None, "mean_abs_error": None}
    difference = (actual - reference).abs()
    return {
        "finite": True,
        "within_tolerance": torch.allclose(actual, reference, **tolerance),
        "exact": torch.equal(actual, reference),
        "max_abs_error": difference.max().item(),
        "mean_abs_error": difference.mean().item(),
    }


@torch.inference_mode()
def encode_clip(text):
    encoded = pipe.tokenizer(
        [text],
        padding="max_length",
        max_length=token_count,
        truncation=True,
        return_tensors="pt",
    )
    attention_mask = (
        encoded.attention_mask.to(device)
        if getattr(pipe.text_encoder.config, "use_attention_mask", False)
        else None
    )
    hidden = pipe.text_encoder(
        input_ids=encoded.input_ids.to(device),
        attention_mask=attention_mask,
    ).last_hidden_state
    if hidden.shape != (1, token_count, hidden_size) or hidden.dtype != dtype or hidden.device != device:
        raise RuntimeError(f"Unexpected CLIP output: {tuple(hidden.shape)}, {hidden.dtype}, {hidden.device}")
    if not torch.isfinite(hidden).all().item():
        raise RuntimeError("The CLIP output contains a nonfinite value.")
    return encoded.input_ids, hidden


negative_ids, negative_embedding = encode_clip(NEGATIVE_PROMPT)
caption_tokens, explicit_embeddings, embedding_checks = [], [], []
with torch.inference_mode():
    for image_id, caption in samples:
        token_ids, positive_embedding = encode_clip(caption)
        internal_positive, internal_negative = pipe.encode_prompt(
            prompt=[caption],
            device=device,
            num_images_per_prompt=1,
            do_classifier_free_guidance=True,
            negative_prompt=[NEGATIVE_PROMPT],
            clip_skip=inference_settings["clip_skip"],
        )
        checks = {
            "positive": tensor_agreement(positive_embedding, internal_positive, embedding_tolerance),
            "negative": tensor_agreement(negative_embedding, internal_negative, embedding_tolerance),
        }
        caption_tokens.append(token_ids)
        explicit_embeddings.append(positive_embedding)
        embedding_checks.append(checks)
        print(f"COCO {image_id:012d} · CLIP agreement {checks}")

print(f"Caption token IDs: {tuple(torch.cat(caption_tokens).shape)}")
print(f"Caption hidden states across five samples: {tuple(torch.cat(explicit_embeddings).shape)}")
print(f"Negative prompt token IDs: {tuple(negative_ids.shape)} · hidden states: {tuple(negative_embedding.shape)}")
print(f"UNet context per guided call: {(2, token_count, hidden_size)}")

## 07 · Generate five paired baselines

For each caption, draw an initial noise tensor with a seeded CPU generator, cast it to FP16, and give each path an independent copy. Each call also gets a freshly seeded generator and a new scheduler built from the same config. The pipeline handles the scheduler's initial noise scaling. We run one image at a time to limit VRAM use.

Capture the final denoised latent before VAE decoding with the pipeline's step callback. Compare the latent, count scheduler steps, and report pixel differences after the normal image and safety processing. Both PNGs and `run.json` are written to Drive after each completed pair, so earlier pairs are available if the session ends. Running this section again makes a new folder.

In [ ]:
from datetime import datetime, timezone
from time import perf_counter


def generate_one(noise, seed, **conditioning):
    scheduler = scheduler_type.from_config(scheduler_config)
    pipe.scheduler = scheduler
    final_latent = None
    denoising_calls = 0

    def capture_latent(_pipeline, _step, _timestep, tensors):
        nonlocal final_latent, denoising_calls
        final_latent = tensors["latents"]
        denoising_calls += 1
        return tensors

    generator = torch.Generator(device="cpu").manual_seed(seed)
    starting_latents = noise.clone()
    torch.cuda.synchronize(device)
    start = perf_counter()
    result = pipe(
        **conditioning,
        **inference_settings,
        latents=starting_latents,
        generator=generator,
        callback_on_step_end=capture_latent,
        callback_on_step_end_tensor_inputs=["latents"],
    )
    torch.cuda.synchronize(device)
    elapsed = perf_counter() - start
    timesteps = scheduler.timesteps.detach().cpu().tolist()
    if final_latent is None or len(timesteps) != denoising_calls or final_latent.shape != latent_shape:
        raise RuntimeError("The final latent or scheduler step count is invalid.")
    flags = result.nsfw_content_detected
    return result.images[0], final_latent, {
        "pipeline_seconds": elapsed,
        "denoising_calls": denoising_calls,
        "timesteps": timesteps,
        "safety_checker_flag": None if flags is None else bool(flags[0]),
    }


started_at = datetime.now(timezone.utc)
run_id = started_at.strftime("%Y%m%dT%H%M%S.%fZ")
run_dir = output_root / run_id
run_dir.mkdir(parents=True, exist_ok=False)
metadata_path = run_dir / "run.json"
run_metadata = {
    "run_id": run_id,
    "started_at_utc": started_at.isoformat(),
    "status": "running",
    "repo_commit": repo_commit,
    "prompt_source": prompt_source,
    "model": {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,
        "dtype": str(dtype),
        "components": components,
        "unet_attention_processors": attention_processors,
    },
    "scheduler": {"class": scheduler_type.__name__, "config": scheduler_config},
    "inference": inference_settings,
    "negative_prompt": NEGATIVE_PROMPT,
    "base_seed": DEFAULT_SEED,
    "noise": {
        "algorithm": "CPU torch.randn float32, then cast to float16",
        "shape": list(latent_shape),
        "sha256_bytes": "contiguous CPU float16 in C order",
        "generator": "fresh CPU torch.Generator with the sample seed for each pipeline call",
    },
    "tolerances": {"clip_embeddings": embedding_tolerance, "final_latents": latent_tolerance},
    "runtime": runtime_info,
    "samples": [],
}


def save_metadata():
    pending = metadata_path.with_suffix(".json.tmp")
    pending.write_text(json.dumps(run_metadata, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    pending.replace(metadata_path)


save_metadata()
for index, (image_id, caption) in enumerate(samples):
    seed = DEFAULT_SEED + index
    cpu_generator = torch.Generator(device="cpu").manual_seed(seed)
    noise_cpu = torch.randn(latent_shape, generator=cpu_generator, device="cpu", dtype=torch.float32).to(dtype)
    noise = noise_cpu.to(device)
    standard_image, standard_latent, standard_info = generate_one(
        noise, seed, prompt=[caption], negative_prompt=[NEGATIVE_PROMPT]
    )
    explicit_image, explicit_latent, explicit_info = generate_one(
        noise, seed, prompt_embeds=explicit_embeddings[index], negative_prompt_embeds=negative_embedding
    )

    latent_check = tensor_agreement(explicit_latent, standard_latent, latent_tolerance)
    equal_timesteps = standard_info["timesteps"] == explicit_info["timesteps"]
    equal_safety_flags = standard_info["safety_checker_flag"] == explicit_info["safety_checker_flag"]
    stem = f"coco_{image_id:012d}_seed_{seed}"
    for image, path_info, label in (
        (standard_image, standard_info, "prompt"),
        (explicit_image, explicit_info, "prompt_embeds"),
    ):
        if image.mode != "RGB" or image.size != (IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f"Unexpected generated image mode or size: {image.mode}, {image.size}")
        path_info["image"] = f"{stem}_{label}.png"
        image.save(run_dir / path_info["image"])

    pixel_difference = np.abs(np.asarray(standard_image, dtype=np.int16) - np.asarray(explicit_image, dtype=np.int16))
    run_metadata["samples"].append({
        "image_id": image_id,
        "caption": caption,
        "seed": seed,
        "initial_noise_sha256": hashlib.sha256(noise_cpu.numpy().tobytes()).hexdigest(),
        "clip_embedding_agreement": embedding_checks[index],
        "final_latent_agreement": latent_check,
        "matching_timesteps": equal_timesteps,
        "matching_safety_flags": equal_safety_flags,
        "pixel_agreement": {
            "exact": not bool(pixel_difference.any()),
            "max_abs_error_0_to_255": int(pixel_difference.max()),
            "mean_abs_error_0_to_255": float(pixel_difference.mean()),
        },
        "standard": standard_info,
        "explicit": explicit_info,
    })
    save_metadata()
    print(f"COCO {image_id:012d} · final latent agreement {latent_check}")
print(f"Saved paired images and runtime metadata in {run_dir}")

## 08 · Verify the saved run

Expect five baseline PNGs from the standard text interface and five companion PNGs from explicit embeddings. Validate each saved image and require positive and negative CLIP agreement, matching scheduler timesteps and safety flags, and final latent agreement for every pair. Pixel errors describe any visible differences. If a numerical check fails, the images and diagnostics remain in Drive for inspection. The tolerances describe agreement within the same Colab session.

In [ ]:
from IPython.display import display

saved = json.loads(metadata_path.read_text(encoding="utf-8"))
if len(saved["samples"]) != len(samples):
    raise RuntimeError(f"The Drive run is incomplete. Open {metadata_path} for completed pairs.")
expected_images = {
    row[path_name]["image"] for row in saved["samples"] for path_name in ("standard", "explicit")
}
if len(expected_images) != 2 * len(samples) or {path.name for path in run_dir.glob("*.png")} != expected_images:
    raise RuntimeError("The Drive run does not contain the ten expected PNG files.")
for filename in sorted(expected_images):
    with Image.open(run_dir / filename) as saved_image:
        if saved_image.format != "PNG" or saved_image.size != (IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f"Unexpected saved format or dimensions: {filename}")
        saved_image.verify()

failed_ids = [
    row["image_id"] for row in saved["samples"]
    if not (
        all(check["within_tolerance"] for check in row["clip_embedding_agreement"].values())
        and row["final_latent_agreement"]["within_tolerance"]
        and row["matching_timesteps"]
        and row["matching_safety_flags"]
    )
]
run_metadata["status"] = "validation_failed" if failed_ids else "complete"
run_metadata["validation"] = {
    "passed": not failed_ids,
    "failed_image_ids": failed_ids,
    "verified_pngs": len(expected_images),
}
run_metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
save_metadata()
print(f"Run metadata: {metadata_path}")
if failed_ids:
    raise AssertionError(f"Conditioning parity failed for COCO IDs {failed_ids}. See run.json.")

preview_size = 192
preview = Image.new("RGB", (2 * preview_size, len(samples) * preview_size), "white")
for index, row in enumerate(saved["samples"]):
    for column, path_name in enumerate(("standard", "explicit")):
        with Image.open(run_dir / row[path_name]["image"]) as image:
            thumbnail = image.resize((preview_size, preview_size), Image.Resampling.LANCZOS)
            preview.paste(thumbnail, (column * preview_size, index * preview_size))
print("PASS · 5 pairs verified. Preview shows standard prompting on the left and explicit conditioning on the right.")
print("Top-to-bottom COCO IDs:", [row["image_id"] for row in saved["samples"]])
display(preview)